In [0]:
df = spark.read.csv("s3://first-bucket-project-2004/raw/data.csv", header=True, inferSchema=True)
df.show()

+--------+----------+-------------+------+-------+-----------+------+
|order_id|order_date|customer_name|region|product|   category|amount|
+--------+----------+-------------+------+-------+-----------+------+
|       1|2024-01-01|         Amit| North| Laptop|Electronics| 50000|
|       2|2024-01-02|        Rahul|  West|  Shirt|   Clothing|  2000|
|       3|2024-01-03|        Sneha| South|  Phone|Electronics| 30000|
|       4|2024-01-04|        Priya|  East|  Shoes|   Footwear|  4000|
|       5|2024-01-05|        Arjun| North| Tablet|Electronics| 20000|
|       6|2024-01-06|         Neha|  West|  Jeans|   Clothing|  3000|
|       7|2024-01-07|        Karan| South|Sandals|   Footwear|  1500|
|       8|2024-01-08|         Riya|  East| Laptop|Electronics| 55000|
+--------+----------+-------------+------+-------+-----------+------+



In [0]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: integer (nullable = true)



In [0]:
from pyspark.sql.functions import when,col

df = df.withColumn(
    "order_value_category",
    when(col("amount") < 5000, "LOW")
    .when((col("amount") >= 5000) & (col("amount") < 20000), "MEDIUM")
    .otherwise("HIGH")
)

In [0]:
df.show()

+--------+----------+-------------+------+-------+-----------+------+--------------------+
|order_id|order_date|customer_name|region|product|   category|amount|order_value_category|
+--------+----------+-------------+------+-------+-----------+------+--------------------+
|       1|2024-01-01|         Amit| North| Laptop|Electronics| 50000|                HIGH|
|       2|2024-01-02|        Rahul|  West|  Shirt|   Clothing|  2000|                 LOW|
|       3|2024-01-03|        Sneha| South|  Phone|Electronics| 30000|                HIGH|
|       4|2024-01-04|        Priya|  East|  Shoes|   Footwear|  4000|                 LOW|
|       5|2024-01-05|        Arjun| North| Tablet|Electronics| 20000|                HIGH|
|       6|2024-01-06|         Neha|  West|  Jeans|   Clothing|  3000|                 LOW|
|       7|2024-01-07|        Karan| South|Sandals|   Footwear|  1500|                 LOW|
|       8|2024-01-08|         Riya|  East| Laptop|Electronics| 55000|                HIGH|

In [0]:
df = df.withColumn(
    "discount",
    when(col("category") == "ELECTRONICS", col("amount") * 0.10)
    .when(col("category") == "CLOTHING", col("amount") * 0.20)
    .otherwise(col("amount") * 0.05)
)

In [0]:
df.show()

+--------+----------+-------------+------+-------+-----------+------+--------------------+--------+
|order_id|order_date|customer_name|region|product|   category|amount|order_value_category|discount|
+--------+----------+-------------+------+-------+-----------+------+--------------------+--------+
|       1|2024-01-01|         Amit| North| Laptop|Electronics| 50000|                HIGH|  2500.0|
|       2|2024-01-02|        Rahul|  West|  Shirt|   Clothing|  2000|                 LOW|   100.0|
|       3|2024-01-03|        Sneha| South|  Phone|Electronics| 30000|                HIGH|  1500.0|
|       4|2024-01-04|        Priya|  East|  Shoes|   Footwear|  4000|                 LOW|   200.0|
|       5|2024-01-05|        Arjun| North| Tablet|Electronics| 20000|                HIGH|  1000.0|
|       6|2024-01-06|         Neha|  West|  Jeans|   Clothing|  3000|                 LOW|   150.0|
|       7|2024-01-07|        Karan| South|Sandals|   Footwear|  1500|                 LOW|    75.0|


In [0]:
df = df.withColumn("final_amount", col("amount") - col("discount"))
df.show()

+--------+----------+-------------+------+-------+-----------+------+--------------------+--------+------------+
|order_id|order_date|customer_name|region|product|   category|amount|order_value_category|discount|final_amount|
+--------+----------+-------------+------+-------+-----------+------+--------------------+--------+------------+
|       1|2024-01-01|         Amit| North| Laptop|Electronics| 50000|                HIGH|  2500.0|     47500.0|
|       2|2024-01-02|        Rahul|  West|  Shirt|   Clothing|  2000|                 LOW|   100.0|      1900.0|
|       3|2024-01-03|        Sneha| South|  Phone|Electronics| 30000|                HIGH|  1500.0|     28500.0|
|       4|2024-01-04|        Priya|  East|  Shoes|   Footwear|  4000|                 LOW|   200.0|      3800.0|
|       5|2024-01-05|        Arjun| North| Tablet|Electronics| 20000|                HIGH|  1000.0|     19000.0|
|       6|2024-01-06|         Neha|  West|  Jeans|   Clothing|  3000|                 LOW|   150

In [0]:
df = df.withColumn(
    "customer_type",
    when(col("amount") > 30000, "PREMIUM").otherwise("REGULAR")
)
df.show()

+--------+----------+-------------+------+-------+-----------+------+--------------------+--------+------------+-------------+
|order_id|order_date|customer_name|region|product|   category|amount|order_value_category|discount|final_amount|customer_type|
+--------+----------+-------------+------+-------+-----------+------+--------------------+--------+------------+-------------+
|       1|2024-01-01|         Amit| North| Laptop|Electronics| 50000|                HIGH|  2500.0|     47500.0|      PREMIUM|
|       2|2024-01-02|        Rahul|  West|  Shirt|   Clothing|  2000|                 LOW|   100.0|      1900.0|      REGULAR|
|       3|2024-01-03|        Sneha| South|  Phone|Electronics| 30000|                HIGH|  1500.0|     28500.0|      REGULAR|
|       4|2024-01-04|        Priya|  East|  Shoes|   Footwear|  4000|                 LOW|   200.0|      3800.0|      REGULAR|
|       5|2024-01-05|        Arjun| North| Tablet|Electronics| 20000|                HIGH|  1000.0|     19000.0

In [0]:
df.write.mode("overwrite").parquet("s3://first-bucket-project-2004/processed/enhanced/")